# 05: model v2

Rule: every choice comes from 5-fold CV on the training set. The test set is opened once, at the very end.
Goal: precision >= 0.80 with the highest recall.

In [1]:
import os
# One math thread per worker: parallel CV overloads the CPU otherwise (same fix as notebook 04).
for var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1"

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

sys.path.insert(0, str(Path("..").resolve()))   # so "from src..." works from notebooks/
from src.thresholds import pick_threshold

DATA_PATH = Path("..") / "data" / "fake_job_postings.csv"
MODEL_DIR = Path("..") / "models"
RESULTS_DIR = Path("..") / "results"
RANDOM_STATE = 42
MIN_PRECISION = 0.80   # my goal, 2026-10-31

In [2]:
df = pd.read_csv(DATA_PATH)
TEXT_COLUMNS = ["title", "company_profile", "description", "requirements", "benefits"]
df["text"] = df[TEXT_COLUMNS].fillna("").agg(" ".join, axis=1)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
df["salary_listed"] = df["salary_range"].notna().astype(int)
df["has_company_profile"] = df["company_profile"].notna().astype(int)
METADATA_COLUMNS = ["has_company_logo", "has_questions", "telecommuting", "salary_listed", "has_company_profile"]

X = df[["text"] + METADATA_COLUMNS]
y = df["fraudulent"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train {len(y_train):,} ({y_train.mean():.2%} fake), Test {len(y_test):,}")
print("Expected (notebook 03): Train 12,852 (4.47%), Test 3,214")
# 🔒 X_test / y_test are not used again until the final section.

Train 12,852 (4.47% fake), Test 3,214
Expected (notebook 03): Train 12,852 (4.47%), Test 3,214


Caveat: v1's settings (C=10, word 1-2 grams) were picked by a grid search on these same 5 folds, so its
out-of-fold numbers are slightly optimistic. v2 is tuned on the same folds, so the comparison stays fair.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
v1 = clone(joblib.load(MODEL_DIR / "job_trust_v1.joblib")["pipeline"])   # same settings, untrained
v1_oof = cross_val_predict(v1, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
print(f"v1 pooled out-of-fold PR-AUC: {average_precision_score(y_train, v1_oof):.3f} "
      "(nb 03 per-fold mean 0.879; pooled differs slightly)")

v1 pooled out-of-fold PR-AUC: 0.878 (nb 03 per-fold mean 0.879; pooled differs slightly)


In [4]:
v1_t = pick_threshold(v1_oof, y_train, min_precision=MIN_PRECISION)
v1_pred = (v1_oof >= v1_t).astype(int)
v1_row = pd.DataFrame([{
    "model": "v1", "threshold": v1_t,
    "precision": precision_score(y_train, v1_pred), "recall": recall_score(y_train, v1_pred),
    "flagged": int(v1_pred.sum()), "false_alarms": int(((v1_pred == 1) & (y_train == 0)).sum()),
}])
RESULTS_DIR.mkdir(exist_ok=True)
v1_row.to_csv(RESULTS_DIR / "v1_cv_at_p80.csv", index=False)
v1_row.round(3)

,model,threshold,precision,recall,flagged,false_alarms
0,v1,0.618,0.8,0.787,565,113


In [5]:
false_alarm = (v1_pred == 1) & (y_train == 0).to_numpy()
fa = df.loc[X_train.index[false_alarm]].assign(v1_prob=v1_oof[false_alarm])
fa = fa.sort_values("v1_prob", ascending=False).head(25)
fa[["v1_prob", "job_id", "title", "location", "salary_range"] + TEXT_COLUMNS[1:]].to_csv(
    Path("..") / "data" / "v1_false_alarms_cv.csv", index=False)
print(f"{false_alarm.sum()} false alarms in total; top 25 saved to data/v1_false_alarms_cv.csv")

113 false alarms in total; top 25 saved to data/v1_false_alarms_cv.csv


## Compare 4 models (training CV only, same 5 folds)

- **A** = character n-grams (3-5 letter chunks like `"$$$"` or `"@gma"`) added next to v1's word TF-IDF.
  Only features a pasted posting can supply, so no train/serve skew.
- **LinearSVC** draws one straight dividing line like LR but scores, not probabilities;
  `CalibratedClassifierCV` turns its scores into 0-1 probabilities so `pick_threshold` works the same way.
- Per fold: recall at precision >= 0.80 (threshold picked on that fold, so a bit optimistic) and PR-AUC.
  "Pooled" = one threshold on all out-of-fold predictions, the same method as v1's 0.787 above.

In [6]:
import tempfile

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


def make_model(char_ngrams: bool, svm: bool, C: float = 1.0) -> Pipeline:
    """v1's features (word 1-2 grams + the 5 yes/no fields), optionally + char 3-5 grams (A), then LR or LinearSVC.
    C only applies to LinearSVC (LR keeps v1's C=10)."""
    features = [("text", TfidfVectorizer(ngram_range=(1, 2), max_features=20000), "text")]
    if char_ngrams:
        # char_wb = chunks taken inside word boundaries; capped so the saved model stays small.
        features.append(("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                                  max_features=50000, sublinear_tf=True), "text"))
    features.append(("meta", "passthrough", METADATA_COLUMNS))
    if svm:
        # Calibration uses an inner 3-fold split of the training part only, never the held-out fold.
        model = CalibratedClassifierCV(LinearSVC(C=C, class_weight="balanced"), cv=3)
    else:
        model = LogisticRegression(class_weight="balanced", C=10, max_iter=1000)   # v1's settings
    return Pipeline([("features", ColumnTransformer(features)), ("model", model)])

In [7]:
MODELS = {
    "v1 (LR)": None,   # out-of-fold predictions already in v1_oof
    "v1+A (LR + char n-grams)": make_model(char_ngrams=True, svm=False),
    "LinearSVC (calibrated)": make_model(char_ngrams=False, svm=True),
    # C = how hard the line tries to fit every training posting (small C = smoother, large C = fits closer).
    **{f"LinearSVC+A C={c}": make_model(char_ngrams=True, svm=True, C=c) for c in [0.01, 0.1, 1, 10]},
}
folds = list(cv.split(X_train, y_train))   # the same 5 folds cross_val_predict uses (fixed random_state)
y_arr = y_train.to_numpy()


def fold_scores(oof):
    """Per fold: recall at precision >= 0.80 (threshold picked inside that fold) and PR-AUC."""
    rec, pr = [], []
    for _, val in folds:
        t = pick_threshold(oof[val], y_arr[val], min_precision=MIN_PRECISION)
        rec.append(recall_score(y_arr[val], oof[val] >= t))
        pr.append(average_precision_score(y_arr[val], oof[val]))
    return np.array(rec), np.array(pr)


def saved_mb(model):
    """Size on disk after fitting on the whole TRAINING set (temp file, not models/)."""
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "m.joblib"
        joblib.dump(clone(model).fit(X_train, y_train), path)
        return path.stat().st_size / 1e6


v1_rec, v1_pr = fold_scores(v1_oof)
rows = []
for name, model in MODELS.items():
    oof = v1_oof if model is None else cross_val_predict(
        model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    rec, pr = fold_scores(oof)
    pooled_t = pick_threshold(oof, y_arr, min_precision=MIN_PRECISION)
    rows.append({
        "model": name,
        "recall@P80 mean": rec.mean(), "recall@P80 std": rec.std(ddof=1),
        "PR-AUC mean": pr.mean(), "PR-AUC std": pr.std(ddof=1),
        "pooled threshold": pooled_t, "pooled recall@P80": recall_score(y_arr, oof >= pooled_t),
        "folds beating v1 (recall)": int((rec > v1_rec).sum()),
        "folds beating v1 (PR-AUC)": int((pr > v1_pr).sum()),
        "saved MB": saved_mb(v1 if model is None else model),
    })

compare = pd.DataFrame(rows)
compare.to_csv(RESULTS_DIR / "cv_v2_compare.csv", index=False)
compare.round(3)

/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


... (5 outputs left out)

/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/Siddarth/job-trust/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,model,recall@P80 mean,recall@P80 std,PR-AUC mean,PR-AUC std,pooled threshold,pooled recall@P80,folds beating v1 (recall),folds beating v1 (PR-AUC),saved MB
0,v1 (LR),0.791,0.033,0.879,0.016,0.618,0.787,0,0,0.975
1,v1+A (LR + char n-grams),0.803,0.036,0.894,0.013,0.545,0.800,4,5,3.088
2,LinearSVC (calibrated),0.791,0.028,0.878,0.016,0.305,0.789,2,2,1.297
3,LinearSVC+A C=0.01,0.315,0.064,0.632,0.035,0.584,0.289,0,0,4.209
4,LinearSVC+A C=0.1,0.772,0.027,0.871,0.017,0.392,0.770,1,1,4.209
5,LinearSVC+A C=1,0.824,0.028,0.897,0.014,0.289,0.819,5,5,4.209
6,LinearSVC+A C=10,0.824,0.026,0.898,0.015,0.272,0.817,4,5,4.209
